[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/19_imbalanced_learning/03_class_weights_threshold_cost.ipynb)

# Bài 03 — Class weight, chọn ngưỡng và học theo chi phí

**🎯 Mục tiêu**
- **Class weight**: tăng trọng số loss của lớp thiểu số — tương đương oversampling "mềm" không tạo dữ liệu.
- **Chọn ngưỡng** theo chi phí trên validation — thường là bước quan trọng nhất.
- So sánh có hệ thống: không làm gì + ngưỡng, class weight, undersampling, SMOTE — theo **chi phí thật**.
- Ensemble cho dữ liệu mất cân bằng (`balanced_subsample`); quy trình khuyến nghị.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/19_imbalanced_learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics as skm
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_fraud_transactions
from ml_course.visualization import set_style

set_style()
fraud = make_fraud_transactions(n=80000, fraud_rate=0.01, seed=31)
features = ["amount_vnd", "hour", "distance_from_home_km", "is_foreign", "n_tx_last_24h", "card_age_days"]
X = fraud[features].astype(float).assign(amount_vnd=lambda d: np.log(d.amount_vnd))
X = X.join(pd.get_dummies(fraud.merchant_category, prefix="cat", dtype=float)).to_numpy()
y = fraud["is_fraud"].to_numpy()
amount = fraud["amount_vnd"].to_numpy()
X_tmp, X_te, y_tmp, y_te, a_tmp, a_te = train_test_split(X, y, amount, test_size=0.3, random_state=0, stratify=y)
X_tr, X_va, y_tr, y_va, a_tr, a_va = train_test_split(X_tmp, y_tmp, a_tmp, test_size=0.3, random_state=0, stratify=y_tmp)
print(f"train {len(y_tr):,} ({y_tr.sum()} gian lận) | valid {len(y_va):,} ({y_va.sum()}) | test {len(y_te):,} ({y_te.sum()})")

## 1. Class weight

Loss có trọng số: $L = \sum_i w_{y_i}\,\ell(y_i, \hat p_i)$. Với `class_weight="balanced"`: $w_c = \frac{n}{K\cdot n_c}$ → lớp thiểu số 1% có trọng số ~50 lần lớp đa số. Tương đương nhân bản mỗi mẫu thiểu số 50 lần (oversampling) nhưng không tạo dữ liệu mới.

In [ ]:
n, n_pos = len(y_tr), y_tr.sum()
print(f"trọng số balanced: lớp 0 = {n / (2 * (n - n_pos)):.3f}, lớp 1 = {n / (2 * n_pos):.2f}")
for cw in [None, "balanced", {0: 1, 1: 10}]:
    m = make_pipeline(StandardScaler(), LogisticRegression(class_weight=cw, max_iter=3000)).fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]
    print(f"class_weight={str(cw):16s} ROC-AUC = {skm.roc_auc_score(y_te, p):.4f} | PR-AUC = {skm.average_precision_score(y_te, p):.4f} | "
          f"recall@0.5 = {skm.recall_score(y_te, p >= 0.5):.3f} | P trung bình = {p.mean():.3f}")

Giống resampling: class weight **dịch** xác suất lên (recall ở ngưỡng 0.5 tăng mạnh) nhưng gần như không đổi khả năng xếp hạng; xác suất không còn calibrated.

## 2. Học theo chi phí: chọn ngưỡng tối ưu

Đặt chi phí thật: kiểm tra một cảnh báo tốn **30 nghìn VND** (FP); bỏ sót một giao dịch gian lận mất **toàn bộ số tiền** giao dịch (FN). Chi phí phụ thuộc từng mẫu → ngưỡng tối ưu cũng có thể phụ thuộc mẫu (bài tập L7). Ở đây chọn một ngưỡng chung trên **validation**:

In [ ]:
COST_CHECK = 30_000


def total_cost(y_true, alert, amounts):
    fp = np.sum(alert & (y_true == 0)) * COST_CHECK
    fn = np.sum(amounts[(~alert) & (y_true == 1)])
    return (fp + fn) / 1e6                                    # triệu VND


def best_threshold(y_true, p, amounts):
    ts = np.unique(np.quantile(p, np.linspace(0.5, 0.9999, 400)))
    costs = [total_cost(y_true, p >= t, amounts) for t in ts]
    return ts[int(np.argmin(costs))]


candidates = {
    "LR, không làm gì": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "LR, class_weight=balanced": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)),
    "HistGB, không làm gì": HistGradientBoostingClassifier(random_state=0),
    "HistGB, class_weight=balanced": HistGradientBoostingClassifier(class_weight="balanced", random_state=0),
    "RF, balanced_subsample": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample", random_state=0),
}
rows = []
for name, m in candidates.items():
    m.fit(X_tr, y_tr)
    p_va, p_te = m.predict_proba(X_va)[:, 1], m.predict_proba(X_te)[:, 1]
    t = best_threshold(y_va, p_va, a_va)
    rows.append({"model": name, "PR-AUC test": skm.average_precision_score(y_te, p_te),
                 "chi phí @0.5": total_cost(y_te, p_te >= 0.5, a_te),
                 "ngưỡng (valid)": t, "chi phí @ngưỡng tối ưu": total_cost(y_te, p_te >= t, a_te)})
baseline_cost = total_cost(y_te, np.zeros_like(y_te, bool), a_te)
res = pd.DataFrame(rows).round(3)
print(f"chi phí nếu không cảnh báo gì: {baseline_cost:,.0f} triệu | cảnh báo tất cả: {total_cost(y_te, np.ones_like(y_te, bool), a_te):,.0f} triệu")
res

Đọc bảng:
- Ở ngưỡng 0.5, các model "không làm gì" trông tệ (bỏ sót nhiều) còn class weight trông tốt — **nhưng** sau khi chọn ngưỡng theo chi phí trên validation, chênh lệch thu hẹp hẳn: phần lớn lợi ích của class weight chỉ là dịch ngưỡng.
- Khác biệt thật giữa các model nằm ở **PR-AUC** (khả năng xếp hạng) — chọn model theo đó, rồi chọn ngưỡng theo chi phí.

In [ ]:
m = candidates["HistGB, không làm gì"]
p_va = m.predict_proba(X_va)[:, 1]
ts = np.unique(np.quantile(p_va, np.linspace(0.5, 0.9999, 300)))
costs = [total_cost(y_va, p_va >= t, a_va) for t in ts]
plt.semilogx(ts, costs)
plt.axvline(ts[int(np.argmin(costs))], color="gray", ls="--", label="ngưỡng tối ưu")
plt.axvline(0.5, color="r", ls=":", label="0.5")
plt.xlabel("ngưỡng")
plt.ylabel("tổng chi phí trên validation (triệu VND)")
plt.legend()
plt.title("Chi phí theo ngưỡng — tối ưu thấp hơn 0.5 rất nhiều")
plt.show()

## 3. Quy trình khuyến nghị

```text
1. Metric đúng (PR-AUC, chi phí, @k) + chia dữ liệu stratified, đủ positive trong valid/test.
2. Model mạnh, không resample: Logistic Regression (baseline) và Gradient Boosting.
3. Chọn model theo PR-AUC / chi phí trên validation (CV).
4. Chọn ngưỡng theo chi phí hoặc năng lực vận hành trên validation.
5. (Tuỳ chọn) class_weight nếu giúp xếp hạng trên CV; resampling chỉ khi có lý do (bài 02).
6. Cần xác suất đúng? → không dùng weight/resample, hoặc hiệu chỉnh prior / calibrate (chương 37).
7. Báo cáo test một lần, kèm CI (ít positive → rất nhiễu).
```

## 🧠 Kiểm tra nhanh

1. `class_weight="balanced"` với 2% positive cho trọng số lớp positive bằng bao nhiêu?
2. Vì sao so sánh các kỹ thuật xử lý mất cân bằng ở ngưỡng 0.5 là không công bằng?
3. Khi chi phí bỏ sót = số tiền giao dịch, ngưỡng tối ưu có nên giống nhau cho mọi giao dịch không?
4. Khi nào nên tránh class weight?

## 📌 Tóm tắt

- Class weight ≈ oversampling mềm: dịch xác suất, ít đổi xếp hạng, làm sai calibration.
- Chọn ngưỡng theo chi phí trên validation là bước có tác động lớn nhất.
- Chọn model theo khả năng xếp hạng (PR-AUC), ngưỡng theo chi phí; resampling là công cụ phụ.

## 📚 Tài liệu tham khảo

- Elkan (2001) — *The Foundations of Cost-Sensitive Learning*
- Chen, Liaw & Breiman (2004) — *Using Random Forest to Learn Imbalanced Data*
- Provost (2000) — *Machine Learning from Imbalanced Data Sets 101*